# Entrenamiento v3 — XGBoost + Grid Search + k-fold CV

En este notebook entrenamos un modelo XGBoost con optimización de hiperparámetros
mediante Grid Search y validación cruzada estratificada (k-fold).

**Experimentos anteriores:**
- Exp 0 (notebook 3): LogReg + TF-IDF → F1-macro val: 0.87
- Exp 1 (notebook 5): LogReg + TF-IDF + features manuales → pendiente

**Este notebook (Exp 2):**
1. TF-IDF + features manuales (misma config)
2. Grid Search con StratifiedKFold (k=5) sobre XGBoost
3. Entrenamiento con mejores hiperparámetros
4. Evaluación en validación
5. Guardar artefactos
6. Registro en MLflow (Experimento 2)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_fusionado"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_dataset_fusionado"
functions._DATASET_TAGS = {"dataset_type": "fusionado", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd() / "data" / "processed"

train_path = data_dir / "train.csv"
val_path   = data_dir / "validation.csv"
test_path  = data_dir / "test.csv"

missing_files = [p.name for p in [train_path, val_path, test_path] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing processed files: {missing_files}. "
        "Please run the preprocessing pipeline to generate them "
        "(e.g., feature engineering scripts) before training."
    )

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val   = val_df["text_final"]
y_val   = val_df["etiqueta"]
X_test  = test_df["text_final"]
y_test  = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

## 2. TF-IDF + Features manuales

In [ ]:
import joblib
from functions import crear_features_manuales, combinar_features

# Cargar el TF-IDF ya ajustado (nb3 lo entrena y guarda con 5 000 términos).
# NO re-ajustamos aquí para que todos los modelos compartan el mismo vectorizador.
tfidf = joblib.load("model/tfidf_vectorizer.joblib")
X_train_tfidf = tfidf.transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF cargado desde model/: {len(tfidf.vocabulary_)} términos")
print(f"Forma train: {X_train_tfidf.shape}")

# Features manuales
feat_train = crear_features_manuales(X_train)
feat_val   = crear_features_manuales(X_val)
feat_test  = crear_features_manuales(X_test)

# Combinadas sparse (TF-IDF + manuales) — se usan para comparar con LogReg
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined   = combinar_features(X_val_tfidf,   feat_val)
X_test_combined  = combinar_features(X_test_tfidf,  feat_test)

print(f"Features totales (sparse): {X_train_combined.shape[1]} "
      f"(TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]})")

## 2b. SVD — comprimir TF-IDF sparse a representación densa para XGBoost

XGBoost trabaja con datos **densos**. Aplicar TruncatedSVD (LSA) convierte los
5000 features TF-IDF sparse en 100 componentes densos que capturan los temas
principales del corpus. Después se concatenan con las features manuales (también densas).

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 100  # componentes LSA

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd   = svd.transform(X_val_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)

# Concatenar componentes SVD (densos) + features manuales (densas)
X_train_xgb = np.hstack([X_train_svd, feat_train.values])
X_val_xgb   = np.hstack([X_val_svd,   feat_val.values])
X_test_xgb  = np.hstack([X_test_svd,  feat_test.values])

varianza = svd.explained_variance_ratio_.sum()
print(f"SVD: {N_COMPONENTS} componentes → {varianza:.1%} varianza explicada")
print(f"Shape train XGBoost: {X_train_xgb.shape}  (denso, vs {X_train_combined.shape} sparse)")

In [ ]:
from functions import grid_search_cv

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
}

# X_train_xgb: SVD(100) + features manuales — todo denso
best_model, best_params, cv_results, le = grid_search_cv(
    X_train_xgb, y_train,
    param_grid=param_grid,
    cv=5,
)

In [ ]:
# Top 10 combinaciones del Grid Search
print("Top 10 combinaciones de hiperparámetros:\n")
cols = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
print(cv_results[cols].head(10).to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report, f1_score

# Evaluar con X_val_xgb (SVD + manuales, denso)
y_val_pred_enc = best_model.predict(X_val_xgb)
y_val_pred = le.inverse_transform(y_val_pred_enc)

print("=== Resultados en VALIDACIÓN (XGBoost + SVD — mejor modelo) ===\n")
print(classification_report(y_val, y_val_pred))

f1_val = f1_score(y_val, y_val_pred, average="macro")
acc_val = (y_val_pred == y_val.values).mean()

print(f"F1-score macro (validación): {f1_val:.4f}")
print(f"Accuracy (validación): {acc_val:.4f}")

In [ ]:
from functions import entrenar_modelo_baseline, crear_tfidf

# Exp 0: LogReg + TF-IDF puro (sin features manuales, sin SVD)
tfidf_solo, X_tr_tfidf0, X_val_tfidf0, _ = crear_tfidf(
    X_train, X_val, X_test, max_features=5000, ngram_range=(1, 2)
)
modelo_exp0 = entrenar_modelo_baseline(X_tr_tfidf0, y_train, X_val_tfidf0, y_val)
EXP0_F1 = f1_score(y_val, modelo_exp0.predict(X_val_tfidf0), average="macro")

# Exp 1: LogReg + TF-IDF + features manuales (sparse)
modelo_exp1 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)
EXP1_F1 = f1_score(y_val, modelo_exp1.predict(X_val_combined), average="macro")

print("\n=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Experimento':<45} {'F1-macro val':>12}")
print("-" * 60)
print(f"{'Exp 0: LogReg + TF-IDF':<45} {EXP0_F1:>12.4f}")
print(f"{'Exp 1: LogReg + TF-IDF + feat. manuales':<45} {EXP1_F1:>12.4f}")
print(f"{'Exp 2: XGBoost + SVD(100) + feat. manuales':<45} {f1_val:>12.4f}")

In [ ]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

# Modelo XGBoost (con label_encoder como atributo)
path_xgb = os.path.join(output_dir, "modelo_xgboost.joblib")
joblib.dump(best_model, path_xgb)
print(f"Modelo XGBoost guardado en:    {path_xgb}")

# tfidf_vectorizer.joblib NO se sobreescribe — es propiedad de nb3
# El SVD sí se guarda porque solo existe en este notebook
path_svd = os.path.join(output_dir, "svd_transformer.joblib")
joblib.dump(svd, path_svd)
print(f"SVD transformer guardado en:   {path_svd}")

# LabelEncoder por separado (por conveniencia)
path_le = os.path.join(output_dir, "label_encoder.joblib")
joblib.dump(le, path_le)
print(f"LabelEncoder guardado en:      {path_le}")

print(f"\nPipeline de predicción: texto → TF-IDF → SVD({N_COMPONENTS}) + features manuales → XGBoost")

# Guardar en mlflow

In [ ]:
# -- MLflow (solo falla esta celda si el servidor no esta disponible) --
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="v3_xgboost_svd_gridsearch"):
        mlflow.log_param("tfidf_max_features",    5000)
        mlflow.log_param("tfidf_ngram_range",     "(1, 2)")
        mlflow.log_param("svd_n_components",      N_COMPONENTS)
        mlflow.log_param("svd_varianza_explicada", round(float(varianza), 4))
        mlflow.log_param("modelo",                "XGBClassifier")
        for param_name, param_val in best_params.items():
            mlflow.log_param(f"xgb_{param_name}", param_val)
        mlflow.log_param("features_manuales",     list(feat_train.columns))
        mlflow.log_param("total_features_xgb",    X_train_xgb.shape[1])
        mlflow.log_param("cv_folds",              5)

        mlflow.log_metric("best_cv_f1_macro",      cv_results.iloc[0]["mean_test_score"])
        mlflow.log_metric("val_f1_macro",           f1_val)
        mlflow.log_metric("val_accuracy",           acc_val)
        mlflow.log_metric("exp0_logreg_f1",         EXP0_F1)
        mlflow.log_metric("exp1_logreg_manual_f1",  EXP1_F1)

        best_cv = cv_results.iloc[0]["mean_test_score"]
        print("Exp 2 (XGBoost + SVD) registrado en MLflow")
        print(f"  Best CV F1-macro: {best_cv:.4f}")
        print(f"  Val F1-macro:     {f1_val:.4f}")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"MLflow no disponible: {e}")
